In [15]:
import json
import re
import csv
import os

In [16]:
os.chdir(r"D:\Kenpath\oan-evaluation")

In [18]:
# Get all categories from qa section
def get_categories(qa_templates):
    categories = {}
    for key in qa_templates.keys():
        category = key.split('.')[0]
        if category not in categories:
            categories[category] = []
        categories[category].append(key)
    return categories


In [19]:
# Replace variables in template with actual values
def substitute_variables(template, variables_dict, num_samples=2):
    # Find variables like [crop], [animal] in the template
    var_pattern = r'\[([^\]]+)\]'
    variables_found = re.findall(var_pattern, template)
    
    if not variables_found:
        return [template] * num_samples
    
    # Get values for each variable
    var_options = {}
    for var in variables_found:
        var_key = var.lower().replace(' ', '_')
        if var_key in variables_dict:
            var_options[var] = variables_dict[var_key][:num_samples]  # Take first N samples
        else:
            var_options[var] = [var]  # Keep as is

        # Generate questions by replacing variables
    questions = []
    if len(var_options) == 1:
        var_name = list(var_options.keys())[0]
        values = var_options[var_name]
        for value in values:
            question = template.replace(f'[{var_name}]', value)
            questions.append(question)
    else:
        # For multiple variables, combine them
        from itertools import product
        combinations = list(product(*var_options.values()))
        for combo in combinations[:num_samples]:
            question = template
            for i, var_name in enumerate(var_options.keys()):
                question = question.replace(f'[{var_name}]', combo[i])
            questions.append(question)
    
    return questions

In [20]:
# Add AgriStack context to questions
def add_agristack_context(question):
    prefixes = [
        "Using AgriStack information, ",
        "Based on AgriStack data, ",
        "With reference to AgriStack, "
    ]
    
    enhanced_questions = []
    for prefix in prefixes:
        base_lower = question.lower().rstrip('?')
        enhanced = f"{prefix}{base_lower}?"
        enhanced_questions.append(enhanced.title())
    
    return enhanced_questions

In [21]:
# Generate all questions and save to CSV
def generate_csv():
    # Load data
    data = load_data()
    variables = data['variables']
    qa_templates = data['qa']
    
    # Get categories
    categories = get_categories(qa_templates)
    
    # Prepare CSV data
    csv_data = []
    
    # Process each category
    for category, qa_keys in categories.items():
        for qa_key in qa_keys:
            template = qa_templates[qa_key]
            
            # Generate questions without AgriStack
            base_questions = substitute_variables(template, variables, 2)
            for question in base_questions:
                csv_data.append({
                    "query": question,
                    "category": category,
                    "agristack_status": "no"
                })
            # Generate questions with AgriStack
            agristack_questions = []
            for base_q in base_questions:
                agristack_questions.extend(add_agristack_context(base_q))
            
            for question in agristack_questions[:2]:  # Take 2 samples
                csv_data.append({
                    "query": question,
                    "category": category,
                    "agristack_status": "yes"
                })
    
    # Save to CSV
    with open('generated_questions.csv', 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['query', 'category', 'agristack_status']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(csv_data)
    
    print(f"Generated {len(csv_data)} questions and saved to generated_questions.csv")

# Run the generation
if __name__ == "__main__":
    generate_csv()

Generated 402 questions and saved to generated_questions.csv
